In [ ]:
using Random
using Statistics
using Printf
using LinearAlgebra
using Plots
using Logging

function find_project_root(start::AbstractString=pwd())
    dir = abspath(start)
    while true
        if isfile(joinpath(dir, "Project.toml")) && isfile(joinpath(dir, "src", "System1D.jl"))
            return dir
        end
        parent = dirname(dir)
        parent == dir && error("Could not locate project root from $start")
        dir = parent
    end
end

PROJECT_ROOT = find_project_root()

if !isdefined(Main, :nb_paths)
    include(joinpath(PROJECT_ROOT, "Experiments", "common", "notebook_helpers.jl"))
end

NOTEBOOK_REL_DIR = joinpath("Experiments", "systems", "two_particle_ho_1d", "dmc", "notebooks")
PATHS = nb_paths(PROJECT_ROOT, NOTEBOOK_REL_DIR)
nb_include_formatting(PATHS.notebook_dir)

if !isdefined(Main, :System1D)
    include(joinpath(PROJECT_ROOT, "src", "System1D.jl"))
end
using .System1D

default(; dpi=170)
nothing


## Model and DMC Parameters

This notebook runs guided DMC for two coupled one-dimensional oscillators with configuration `R = (x1, x2)` and Hamiltonian
`H = -D (d^2/dx1^2 + d^2/dx2^2) + 0.5 * omega^2 * (x1^2 + x2^2) + 0.5 * kappa * (x1 - x2)^2`.

The guiding state is written in center-of-mass and relative coordinates.

Parameters used below:
- Oscillator frequency `omega = 1.0`
- Coupling `kappa = 0.7`
- Time step `dt = 5.0e-3`
- Total steps `nsteps = 400`
- Equilibration steps `nequil = 60`
- Target population `targetN = 3000`

Trial / node structure:
- Guiding policy: `ImportanceGuiding(trial, H)`
- Node policy: `NoNode()`


## Julia Construction

The next cell defines the coupled Hamiltonian, the normal-mode trial state, the DMC parameters, and the notebook toggles.

The run cell after it executes the simulation, and the last cell turns the stored state into plots.


In [ ]:
omega = 1.0
kappa = 0.7

V(R) = begin
    x1, x2 = R
    0.5 * omega^2 * (x1^2 + x2^2) + 0.5 * kappa * (x1 - x2)^2
end
H = Hamiltonian(2, 0.5, V)

omega_rel = sqrt(omega^2 + 2 * kappa)
logpsi(R) = begin
    x1, x2 = R
    S = x1 + x2
    Delta = x1 - x2
    -0.25 * omega * S^2 - 0.25 * omega_rel * Delta^2
end
gradlogpsi(R) = begin
    x1, x2 = R
    S = x1 + x2
    Delta = x1 - x2
    [
        -0.5 * (omega * S + omega_rel * Delta),
        -0.5 * (omega * S - omega_rel * Delta),
    ]
end
lapllogpsi(R) = -(omega + omega_rel)
trial = TrialWF(logpsi, gradlogpsi, lapllogpsi)
guiding = ImportanceGuiding(trial, H)

targetN = 3000
dt = 5.0e-3
nsteps = 400
nequil = 60
ET0 = 0.5 * (omega + omega_rel)
branch_cap = 10
nblocks = 50

params = DMCParams(; dt=dt, nsteps=nsteps, nequil=nequil, targetN=targetN, ET0=ET0, population_control_gain=1.0, branch_cap=branch_cap, nblocks=nblocks)

rng_init = MersenneTwister(1234)
initial_positions = [2 .* rand(rng_init, 2) .- 1 for _ in 1:targetN]

SNAPSHOT_STEPS = nb_default_snapshot_steps(nsteps)
NBINS = 140
DENSITY_SMOOTHING = 9

RUN_LABEL = "guided"
RUN_COLOR = :navy
PLOT_TITLE = "Coupled two-particle HO DMC"

SHOW_PROGRESS = false
PROGRESS_EVERY = 0
DEBUG_MODE = false
DEBUG_EVERY = 20
WRITE_RUN_CSV = false
CSV_FILENAME = "two_particle_ho_dmc.csv"
SAVE_FIGURES = false
FIGURE_STEM = "two_particle_ho_dmc"


In [ ]:
sim = run_dmc(
    H,
    params,
    initial_positions;
    rng=MersenneTwister(42),
    guiding=guiding,
    snapshot_steps=SNAPSHOT_STEPS,
    show_progress=SHOW_PROGRESS,
    progress_every=PROGRESS_EVERY,
    progress_label=RUN_LABEL,
    debug=DEBUG_MODE,
    debug_every=DEBUG_EVERY,
)

start_idx = min(params.nequil + 1, length(sim.energy_mean_history))
mean_energy, sem_energy = nb_mean_sem(sim.energy_mean_history[start_idx:end])

println(@sprintf("%s mean energy after nequil=%d: %.8f +/- %.3e", RUN_LABEL, params.nequil, mean_energy, sem_energy))
println("final walker population = ", sim.population_history[end])

if WRITE_RUN_CSV
    csv_path = joinpath(PATHS.tables_dir, CSV_FILENAME)
    nb_write_csv(csv_path, nb_dmc_rows(RUN_LABEL, sim))
    println("Wrote run CSV to: ", abspath(csv_path))
end


In [ ]:
history_fig = nb_plot_dmc_history([sim]; labels=[RUN_LABEL], colors=[RUN_COLOR], title_prefix=PLOT_TITLE)
display(history_fig)
nb_save_figure(history_fig, PATHS.figures_dir, FIGURE_STEM, "history"; enabled=SAVE_FIGURES)

final_snapshot = nb_last_snapshot(sim)
x1 = Float64[R[1] for R in final_snapshot]
x2 = Float64[R[2] for R in final_snapshot]
coord_limits = nb_padded_limits(vcat(x1, x2); pad_frac=0.10)

scatter_panel = scatter(
    x1,
    x2;
    xlabel="x1",
    ylabel="x2",
    title="Final walker cloud",
    markersize=2.3,
    alpha=0.35,
    color=:navy,
    label=false,
    xlims=coord_limits,
    ylims=coord_limits,
)

marginal_panel = plot(
    xlabel="coordinate",
    ylabel="density",
    title="Final coordinate marginals",
    legend=:topright,
    xlims=coord_limits,
)
for (coord_idx, label, color) in ((1, "x1", :navy), (2, "x2", :darkorange))
    centers, density = nb_density_curve_from_snapshot(
        final_snapshot;
        coord=coord_idx,
        nbins=NBINS,
        xmin=coord_limits[1],
        xmax=coord_limits[2],
        smoothing_window=DENSITY_SMOOTHING,
    )
    plot!(marginal_panel, centers, density; label=label, color=color, linewidth=2.3)
end

state_fig = plot(scatter_panel, marginal_panel; layout=(1, 2), size=(1200, 500))
display(state_fig)
nb_save_figure(state_fig, PATHS.figures_dir, FIGURE_STEM, "state"; enabled=SAVE_FIGURES)
